# Debug TextVQA Samples From Qwen2-VL MoE LoRA Checkpoint

This notebook loads the already-trained MoE LoRA adapters from the same checkpoint path used by `train_qwen2vl_moe_lora.ipynb`, then prints TextVQA test samples with router choice, prediction, ground-truth answers, and Exact Match.

Checkpoint root used by the MoE notebook:

- Colab/Drive: `/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/qwen2vl_moe_lora`
- Local fallback: `outputs/checkpoints/qwen2vl_moe_lora`

## 1. Colab Repository Setup

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)
    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

In [ ]:
from pathlib import Path
from collections import Counter
import gc
import json
import os
import random
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

required_packages = ["qwen-vl-utils", "accelerate", "peft", "scikit-learn"]
for package_name in required_packages:
    import_name = package_name.replace("-", "_")
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=True)

import torch

try:
    import torchao
    torchao_version = getattr(torchao, "__version__", "0.0.0")
    major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
    if (major, minor) < (0, 16):
        print(f"Removing incompatible torchao=={torchao_version} before importing PEFT.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
        for module_name in list(sys.modules):
            if module_name == "torchao" or module_name.startswith("torchao."):
                del sys.modules[module_name]
except ModuleNotFoundError:
    pass

from peft import PeftModel
from qwen_vl_utils import process_vision_info
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

from src.evaluation.metrics import exact_match, mean_score
from src.models.lora_adapter import QWEN2VL_LORA_EXPERTS
from src.routing.task_router import RouterDecision, format_router_decision

torch.__version__

## 3. Config

In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

LOCAL_CHECKPOINT_ROOT = PROJECT_ROOT / "outputs/checkpoints/qwen2vl_moe_lora"
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/qwen2vl_moe_lora")
USE_GOOGLE_DRIVE_CHECKPOINT = True

MAX_SAMPLES = 5000
TRAIN_RATIO = 0.8
TEST_LIMIT = 1000
TEXTVQA_PRINT_LIMIT = 30
SEED = 42
MAX_NEW_TOKENS = 32
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

DATA_LIMITS = {"docvqa": 1667, "chartqa": 1667, "textvqa": 1666}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("DEVICE:", DEVICE)
print("TORCH_DTYPE:", TORCH_DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Checkpoint Location

In [ ]:
CHECKPOINT_ROOT = LOCAL_CHECKPOINT_ROOT

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_ROOT = DRIVE_CHECKPOINT_ROOT
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        CHECKPOINT_ROOT = LOCAL_CHECKPOINT_ROOT

EXPERT_CHECKPOINT_DIRS = {
    task_type: CHECKPOINT_ROOT / expert.adapter_name
    for task_type, expert in QWEN2VL_LORA_EXPERTS.items()
}

print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
for task_type, checkpoint_dir in EXPERT_CHECKPOINT_DIRS.items():
    print(task_type, "->", checkpoint_dir, "exists=", checkpoint_dir.exists())
    assert checkpoint_dir.exists(), f"Missing checkpoint for {task_type}: {checkpoint_dir}"

## 5. Prepare Data If Needed

In [ ]:
def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


current_count = count_jsonl(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    build_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/build_multitask_data.py"),
        "--split",
        "validation",
    ]
    print("Running:", " ".join(build_command))
    subprocess.run(build_command, check=True, cwd=PROJECT_ROOT)

print(f"Processed examples after setup: {count_jsonl(METADATA_PATH)}")

## 6. Load Records With Same Split as MoE Training

In [ ]:
TASK_ALIASES = {
    "chart_qa": "chartqa",
    "chartqa": "chartqa",
    "document_qa": "docvqa",
    "docvqa": "docvqa",
    "scene_text_qa": "textvqa",
    "textvqa": "textvqa",
}


def canonical_task_type(record: dict) -> str:
    raw_task = str(record.get("task_type", "")).lower()
    dataset = str(record.get("dataset", "")).lower()
    return TASK_ALIASES.get(raw_task) or TASK_ALIASES.get(dataset) or "textvqa"


random.seed(SEED)
torch.manual_seed(SEED)

all_records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        record["canonical_task_type"] = canonical_task_type(record)
        all_records.append(record)

all_records_by_task = {
    task_type: [record for record in all_records if record["canonical_task_type"] == task_type]
    for task_type in QWEN2VL_LORA_EXPERTS
}
for task_records in all_records_by_task.values():
    random.shuffle(task_records)

per_task_limit = max(1, MAX_SAMPLES // len(QWEN2VL_LORA_EXPERTS))
records = []
for task_type in ("chartqa", "docvqa", "textvqa"):
    records.extend(all_records_by_task[task_type][:per_task_limit])

remaining_records = [
    record
    for task_records in all_records_by_task.values()
    for record in task_records[per_task_limit:]
]
random.shuffle(remaining_records)
records.extend(remaining_records[: max(0, MAX_SAMPLES - len(records))])
random.shuffle(records)

split_index = int(len(records) * TRAIN_RATIO)
train_records = records[:split_index]
test_records = records[split_index:]

print("Total records:", len(records))
print("Train records:", len(train_records), Counter(record["canonical_task_type"] for record in train_records))
print("Test records:", len(test_records), Counter(record["canonical_task_type"] for record in test_records))

## 7. Qwen Helpers

In [ ]:
def free_gpu_memory() -> None:
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def get_model_device(model: torch.nn.Module) -> torch.device:
    return next(model.parameters()).device


def load_processor() -> AutoProcessor:
    return AutoProcessor.from_pretrained(
        MODEL_NAME,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )


def load_qwen_backbone() -> Qwen2VLForConditionalGeneration:
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=TORCH_DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    model.config.use_cache = False
    if DEVICE != "cuda":
        model.to(DEVICE)
    return model


ANSWER_FORMAT_INSTRUCTION = (
    "Answer only the direct answer. "
    "Keep all necessary words or numbers. "
    "Do not explain. "
    "If the question is unclear, malformed, not a question, or cannot be answered from readable image text, answer unanswerable."
)


def format_question_prompt(question: str) -> str:
    return f"{question}\n\n{ANSWER_FORMAT_INSTRUCTION}"


def build_messages(image_path: str, question: str) -> list[dict]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": format_question_prompt(question)},
            ],
        }
    ]

## 8. Train Lightweight Router

In [ ]:
router_questions = [record["question"] for record in train_records]
router_labels = [record["canonical_task_type"] for record in train_records]
label_counts = Counter(router_labels)
can_stratify = len(label_counts) == 3 and min(label_counts.values()) >= 2

train_questions, eval_questions, train_labels, eval_labels = train_test_split(
    router_questions,
    router_labels,
    test_size=0.2,
    random_state=SEED,
    stratify=router_labels if can_stratify else None,
)

router_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)
router_model.fit(train_questions, train_labels)

eval_predictions = router_model.predict(eval_questions)
router_accuracy = accuracy_score(eval_labels, eval_predictions)
print(f"Router eval accuracy: {router_accuracy:.4f}")
print(classification_report(eval_labels, eval_predictions))


def learned_route_expert(question: str) -> tuple[RouterDecision, dict[str, float]]:
    probabilities = router_model.predict_proba([question])[0]
    classes = list(router_model.classes_)
    prob_by_task = {task: float(prob) for task, prob in zip(classes, probabilities)}
    task_type = classes[int(probabilities.argmax())]
    expert = QWEN2VL_LORA_EXPERTS[task_type]

    decision = RouterDecision(
        task_type=task_type,
        expert_id=expert.expert_id,
        adapter_name=expert.adapter_name,
        checkpoint_dir=expert.checkpoint_dir,
        confidence=prob_by_task[task_type],
    )
    return decision, prob_by_task

## 9. Load MoE LoRA Checkpoints

In [ ]:
free_gpu_memory()
processor = load_processor()
base_model = load_qwen_backbone()

first_task = "chartqa"
model = PeftModel.from_pretrained(
    base_model,
    EXPERT_CHECKPOINT_DIRS[first_task],
    adapter_name=first_task,
)

for task_type in ("docvqa", "textvqa"):
    model.load_adapter(EXPERT_CHECKPOINT_DIRS[task_type], adapter_name=task_type)

if DEVICE != "cuda":
    model.to(DEVICE)
model.eval()
print("Loaded adapters:", list(model.peft_config.keys()))

## 10. Print TextVQA Samples

In [ ]:
def qwen_generate_with_selected_adapter(example: dict, adapter_name: str) -> str:
    model.set_adapter(adapter_name)
    messages = build_messages(example["image_path"], example["question"])
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(get_model_device(model))

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            use_cache=False,
        )

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(generated_trimmed, skip_special_tokens=True)[0].strip()


textvqa_examples = [
    record
    for record in test_records[:TEST_LIMIT]
    if record["canonical_task_type"] == "textvqa"
]

textvqa_debug_records = []
for index, example in enumerate(textvqa_examples[:TEXTVQA_PRINT_LIMIT], start=1):
    decision, router_probs = learned_route_expert(example["question"])
    prediction = qwen_generate_with_selected_adapter(example, decision.task_type)
    score = exact_match(prediction, example["answers"])

    textvqa_debug_records.append({
        "index": index,
        "question": example["question"],
        "answers": example["answers"],
        "prediction": prediction,
        "exact_match": score,
        "expert_id": decision.expert_id,
        "adapter_name": decision.adapter_name,
        "router_probs": router_probs,
    })

    print(f"TextVQA sample {index}/{min(TEXTVQA_PRINT_LIMIT, len(textvqa_examples))}")
    print(format_router_decision(decision, sample_id=index, true_task_type="textvqa"))
    print("router_probs:", {task: round(prob, 3) for task, prob in router_probs.items()})
    print("question:", example["question"])
    print("prediction:", prediction)
    print("answers:", example["answers"])
    print("exact_match:", score)
    print("-" * 80)

print("Printed TextVQA samples:", len(textvqa_debug_records))
print("Sample TextVQA Exact Match:", round(mean_score([record["exact_match"] for record in textvqa_debug_records]), 4))
textvqa_debug_records[:3]